# 05_Descriptive Statistics

This notebook generates comprehensive descriptive statistics for the Persuade dataset across different demographic groups (SES, Race, Gender). It includes mean values, standard deviations, standard errors, and gap analyses.

## Cell 1: Imports and Configuration

In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [2]:
# Set output directory for tables
output_dir = "../outputs/tables"
import os
os.makedirs(output_dir, exist_ok=True)

## Cell 2: Load Data and Preprocessing

In [8]:
# Load Data
df = pd.read_csv("../data/processed/original/original_full.csv")

# --- Preprocessing ---

# 1. SES: Low vs High
# economically_disadvantaged_1 = Low SES (Free/Reduced Lunch)
# economically_disadvantaged_0 = High SES (Likely Paid Lunch/Not Disadvantaged)
df['ses_group'] = df['economically_disadvantaged_1'].apply(lambda x: 'Low SES' if x == 1 else 'High SES')

# 2. Race: White vs Non-White
df['race_group'] = df['race_ethnicity_White'].apply(lambda x: 'White' if x == 1 else 'Non-White')

# 3. Gender
# Assuming gender_F is reliable for Female.
def get_gender(row):
    if row['gender_F'] == 1:
        return 'Female'
    else:
        return 'Male'
        
df['gender_group'] = df.apply(get_gender, axis=1)

print(f"Total essays: {len(df)}")
print(f"\nSES Distribution:\n{df['ses_group'].value_counts()}")
print(f"\nRace Distribution:\n{df['race_group'].value_counts()}")
print(f"\nGender Distribution:\n{df['gender_group'].value_counts()}")

Total essays: 20759

SES Distribution:
ses_group
High SES    11116
Low SES      9643
Name: count, dtype: int64

Race Distribution:
race_group
Non-White    11371
White         9388
Name: count, dtype: int64

Gender Distribution:
gender_group
Female    10626
Male      10133
Name: count, dtype: int64


## Cell 3: Define Metrics and Descriptions

In [9]:
# --- Selection of Metrics ---
# We use existing columns from the dataset instead of calculating new ones.

# Dictionary mapping column names to readable labels/descriptions
metric_descriptions = {
    'holistic_essay_score': 'Holistic Essay Score (Outcome)',
    
    # Demographic Indicators
    'gender_F': 'Female',
    'race_ethnicity_White': 'White',
    
    # Grade Level
    'grade_level_6.0': '6th Grade',
    'grade_level_8.0': '8th Grade',
    'grade_level_9.0': '9th Grade',
    'grade_level_10.0': '10th Grade',
    'grade_level_11.0': '11th Grade',
    'grade_level_12.0': '12th Grade',
    
    # Length / Volume Metrics
    'taassc_nwords': 'Word Count (Volume)',
    
    # Lexical Complexity (Reading Level Proxies)
    'taassc_wrd_length': 'Avg Word Length (Lexical Sophistication)',
    
    # Syntactic Complexity (Style)
    'taassc_mlc': 'Mean Length of Clause (Syntactic Complexity)',
    'taassc_mltu': 'Mean Length of T-Unit (Sentence Complexity)',
    'taassc_mean_verbal_deps': 'Mean Verbal Dependencies',
    'taassc_infinitive_prop': 'Proportion of Infinitives',
    'taassc_nonfinite_prop': 'Proportion of Non-Finite Clauses',
    
    # Advanced Lexical Metrics
    'taaled_lexical_density_tokens': 'Lexical Density (tokens)',
    'taaled_mtld_original_aw': 'Lexical Diversity (MTLD; all words)',
    'taaco_lemma_mattr': 'Lexical Diversity (MATTR; lemmas)',
    
    # Cohesion Metrics
    'taaco_adjacent_overlap_cw_sent': 'Content-word Overlap (adjacent sentences)',
    'taaco_word2vec_1_all_sent': 'Semantic Cohesion (Word2Vec; adjacent sentences)',
    'taaco_basic_connectives': 'Connectives (basic)',
    'taaco_reason_and_purpose': 'Causal Connectives (reason & purpose)',
    
    # Style Markers
    'taassc_nominalization': 'Nominalizations',
    'taaco_pronoun_noun_ratio': 'Pronoun-to-noun Ratio',
}

target_metrics = list(metric_descriptions.keys())

print("Data loaded and groups defined. Using metrics:")
for m, desc in metric_descriptions.items():
    print(f" - {m}: {desc}")

Data loaded and groups defined. Using metrics:
 - holistic_essay_score: Holistic Essay Score (Outcome)
 - gender_F: Female
 - race_ethnicity_White: White
 - grade_level_6.0: 6th Grade
 - grade_level_8.0: 8th Grade
 - grade_level_9.0: 9th Grade
 - grade_level_10.0: 10th Grade
 - grade_level_11.0: 11th Grade
 - grade_level_12.0: 12th Grade
 - taassc_nwords: Word Count (Volume)
 - taassc_wrd_length: Avg Word Length (Lexical Sophistication)
 - taassc_mlc: Mean Length of Clause (Syntactic Complexity)
 - taassc_mltu: Mean Length of T-Unit (Sentence Complexity)
 - taassc_mean_verbal_deps: Mean Verbal Dependencies
 - taassc_infinitive_prop: Proportion of Infinitives
 - taassc_nonfinite_prop: Proportion of Non-Finite Clauses
 - taaled_lexical_density_tokens: Lexical Density (tokens)
 - taaled_mtld_original_aw: Lexical Diversity (MTLD; all words)
 - taaco_lemma_mattr: Lexical Diversity (MATTR; lemmas)
 - taaco_adjacent_overlap_cw_sent: Content-word Overlap (adjacent sentences)
 - taaco_word2vec_

## Cell 4: Statistical Analysis Functions

In [10]:
# Statistical helper functions

def mean_se(sub_df, metrics):
    """Calculate mean, SD, and SE for metrics in a subset of data."""
    n = len(sub_df)
    mean = sub_df[metrics].mean()
    sd = sub_df[metrics].std(ddof=1)
    se = sd / np.sqrt(n)
    return n, mean, sd, se

def fmt_cell(mean, se, digits=3):
    """Format cell as: mean (SE) for LaTeX output using shortstack."""
    return rf"\shortstack{{{mean:.{digits}f} \\ ({se:.{digits}f})}}"

def get_group_stats(df, group_col, group_label_map, metrics):
    """Calculate stats for each group."""
    results = {}
    
    for group_val, display_name in group_label_map.items():
        sub_df = df[df[group_col] == group_val]
        n = len(sub_df)
        stats = sub_df[metrics].mean().to_dict()
        
        # Add relevant metadata
        col_data = {'N': n}
        for m in metrics:
            col_data[f"{m}_mean"] = stats[m]
            col_data[f"{m}_sd"] = sub_df[metrics].std()[m]
            
        results[display_name] = col_data
        
    return pd.DataFrame(results)

## Cell 5: Compute Descriptive Statistics

In [11]:
# --- Compute statistics ---

# Full sample statistics
nF, meanF, sdF, seF = mean_se(df, target_metrics)

# SES-based breakdown
low_df  = df[df['ses_group'] == 'Low SES']
high_df = df[df['ses_group'] == 'High SES']

nL, meanL, sdL, seL = mean_se(low_df, target_metrics)
nH, meanH, sdH, seH = mean_se(high_df, target_metrics)

# Race-based breakdown
white_df = df[df['race_group'] == 'White']
nonwhite_df = df[df['race_group'] == 'Non-White']

nW, meanW, sdW, seW = mean_se(white_df, target_metrics)
nNW, meanNW, sdNW, seNW = mean_se(nonwhite_df, target_metrics)

# Gender-based breakdown
female_df = df[df['gender_group'] == 'Female']
male_df = df[df['gender_group'] == 'Male']

nFem, meanFem, sdFem, seFem = mean_se(female_df, target_metrics)
nMale, meanMale, sdMale, seMale = mean_se(male_df, target_metrics)

# Calculate gaps (differences between groups)
gap_ses_mean = meanH - meanL
gap_ses_se = np.sqrt((sdH**2)/nH + (sdL**2)/nL)  # SE of difference

gap_race_mean = meanW - meanNW
gap_race_se = np.sqrt((sdW**2)/nW + (sdNW**2)/nNW)

gap_gender_mean = meanFem - meanMale
gap_gender_se = np.sqrt((sdFem**2)/nFem + (sdMale**2)/nMale)

print("\nDescriptive Statistics Summary:")
print(f"Full Sample: N = {nF}")
print(f"Low SES: N = {nL}, High SES: N = {nH}")
print(f"White: N = {nW}, Non-White: N = {nNW}")
print(f"Female: N = {nFem}, Male: N = {nMale}")


Descriptive Statistics Summary:
Full Sample: N = 20759
Low SES: N = 9643, High SES: N = 11116
White: N = 9388, Non-White: N = 11371
Female: N = 10626, Male: N = 10133


## Cell 6: Build and Display Summary Tables

In [12]:
# --- Build table with pretty row labels ---

row_labels = ["N"] + [metric_descriptions.get(m, m) for m in target_metrics]
final_table = pd.DataFrame(index=row_labels)

# Add each column group
final_table["Full Sample"] = [str(nF)] + [fmt_cell(meanF[m], seF[m]) for m in target_metrics]
final_table["Low SES"]     = [str(nL)] + [fmt_cell(meanL[m], seL[m]) for m in target_metrics]
final_table["High SES"]    = [str(nH)] + [fmt_cell(meanH[m], seH[m]) for m in target_metrics]
final_table["Gap SES (High-Low)"] = [""] + [fmt_cell(gap_ses_mean[m], gap_ses_se[m]) for m in target_metrics]

final_table["Non-White"]   = [str(nNW)] + [fmt_cell(meanNW[m], seNW[m]) for m in target_metrics]
final_table["White"]       = [str(nW)]  + [fmt_cell(meanW[m], seW[m]) for m in target_metrics]
final_table["Gap Race (White-NonWhite)"] = [""] + [fmt_cell(gap_race_mean[m], gap_race_se[m]) for m in target_metrics]

final_table["Male"]        = [str(nMale)] + [fmt_cell(meanMale[m], seMale[m]) for m in target_metrics]
final_table["Female"]      = [str(nFem)]  + [fmt_cell(meanFem[m], seFem[m]) for m in target_metrics]
final_table["Gap Gender (Female-Male)"] = [""] + [fmt_cell(gap_gender_mean[m], gap_gender_se[m]) for m in target_metrics]

# Reorder columns for display
cols_order = [
    "Full Sample",
    "Low SES", "High SES", "Gap SES (High-Low)",
    "Non-White", "White", "Gap Race (White-NonWhite)",
    "Male", "Female", "Gap Gender (Female-Male)"
]
final_table = final_table[cols_order]

print("\nFinal Descriptive Statistics Table (with SEs):")
display(final_table)


Final Descriptive Statistics Table (with SEs):


,Full Sample,Low SES,High SES,Gap SES (High-Low),Non-White,White,Gap Race (White-NonWhite),Male,Female,Gap Gender (Female-Male)
N,20759,9643,11116,,11371,9388,,10133,10626,
Holistic Essay Score (Outcome),\shortstack{3.337 \\ (0.008)},\shortstack{2.979 \\ (0.011)},\shortstack{3.648 \\ (0.011)},\shortstack{0.669 \\ (0.016)},\shortstack{3.224 \\ (0.011)},\shortstack{3.473 \\ (0.012)},\shortstack{0.249 \\ (0.016)},\shortstack{3.219 \\ (0.012)},\shortstack{3.449 \\ (0.011)},\shortstack{0.230 \\ (0.016)}
Female,\shortstack{0.512 \\ (0.003)},\shortstack{0.526 \\ (0.005)},\shortstack{0.500 \\ (0.005)},\shortstack{-0.026 \\ (0.007)},\shortstack{0.522 \\ (0.005)},\shortstack{0.500 \\ (0.005)},\shortstack{-0.022 \\ (0.007)},\shortstack{0.000 \\ (0.000)},\shortstack{1.000 \\ (0.000)},\shortstack{1.000 \\ (0.000)}
White,\shortstack{0.452 \\ (0.003)},\shortstack{0.247 \\ (0.004)},\shortstack{0.630 \\ (0.005)},\shortstack{0.384 \\ (0.006)},\shortstack{0.000 \\ (0.000)},\shortstack{1.000 \\ (0.000)},\shortstack{1.000 \\ (0.000)},\shortstack{0.463 \\ (0.005)},\shortstack{0.442 \\ (0.005)},\shortstack{-0.022 \\ (0.007)}
6th Grade,\shortstack{0.066 \\ (0.002)},\shortstack{0.073 \\ (0.003)},\shortstack{0.060 \\ (0.002)},\shortstack{-0.012 \\ (0.003)},\shortstack{0.056 \\ (0.002)},\shortstack{0.078 \\ (0.003)},\shortstack{0.022 \\ (0.004)},\shortstack{0.062 \\ (0.002)},\shortstack{0.070 \\ (0.002)},\shortstack{0.007 \\ (0.003)}
8th Grade,\shortstack{0.461 \\ (0.003)},\shortstack{0.421 \\ (0.005)},\shortstack{0.495 \\ (0.005)},\shortstack{0.075 \\ (0.007)},\shortstack{0.424 \\ (0.005)},\shortstack{0.505 \\ (0.005)},\shortstack{0.081 \\ (0.007)},\shortstack{0.448 \\ (0.005)},\shortstack{0.473 \\ (0.005)},\shortstack{0.026 \\ (0.007)}
9th Grade,\shortstack{0.001 \\ (0.000)},\shortstack{0.001 \\ (0.000)},\shortstack{0.001 \\ (0.000)},\shortstack{0.000 \\ (0.000)},\shortstack{0.001 \\ (0.000)},\shortstack{0.000 \\ (0.000)},\shortstack{-0.001 \\ (0.000)},\shortstack{0.001 \\ (0.000)},\shortstack{0.000 \\ (0.000)},\shortstack{-0.001 \\ (0.000)}
10th Grade,\shortstack{0.304 \\ (0.003)},\shortstack{0.376 \\ (0.005)},\shortstack{0.242 \\ (0.004)},\shortstack{-0.134 \\ (0.006)},\shortstack{0.327 \\ (0.004)},\shortstack{0.277 \\ (0.005)},\shortstack{-0.050 \\ (0.006)},\shortstack{0.312 \\ (0.005)},\shortstack{0.297 \\ (0.004)},\shortstack{-0.015 \\ (0.006)}
11th Grade,\shortstack{0.149 \\ (0.002)},\shortstack{0.101 \\ (0.003)},\shortstack{0.190 \\ (0.004)},\shortstack{0.089 \\ (0.005)},\shortstack{0.158 \\ (0.003)},\shortstack{0.137 \\ (0.004)},\shortstack{-0.020 \\ (0.005)},\shortstack{0.157 \\ (0.004)},\shortstack{0.141 \\ (0.003)},\shortstack{-0.016 \\ (0.005)}
12th Grade,\shortstack{0.019 \\ (0.001)},\shortstack{0.029 \\ (0.002)},\shortstack{0.011 \\ (0.001)},\shortstack{-0.018 \\ (0.002)},\shortstack{0.034 \\ (0.002)},\shortstack{0.002 \\ (0.000)},\shortstack{-0.031 \\ (0.002)},\shortstack{0.020 \\ (0.001)},\shortstack{0.019 \\ (0.001)},\shortstack{-0.001 \\ (0.002)}


## Cell 7: Export Results

In [13]:
# --- Export to LaTeX ---
latex_str = final_table.to_latex(
    escape=False,  # keep \shortstack as is
    index=True
)

latex_path = f"{output_dir}/descriptive_statistics_summary.tex"
with open(latex_path, "w") as f:
    f.write(latex_str)

# --- Export numeric table to CSV (raw means and SDs) ---
# Create a simplified version for CSV export
csv_table = pd.DataFrame()
csv_table['Metric'] = row_labels
csv_table['Full_Sample_N'] = nF
csv_table['Full_Sample_Mean'] = [np.nan] + [meanF[m] for m in target_metrics]
csv_table['Full_Sample_SD'] = [np.nan] + [sdF[m] for m in target_metrics]

csv_table['Low_SES_N'] = nL
csv_table['Low_SES_Mean'] = [np.nan] + [meanL[m] for m in target_metrics]
csv_table['Low_SES_SD'] = [np.nan] + [sdL[m] for m in target_metrics]

csv_table['High_SES_N'] = nH
csv_table['High_SES_Mean'] = [np.nan] + [meanH[m] for m in target_metrics]
csv_table['High_SES_SD'] = [np.nan] + [sdH[m] for m in target_metrics]

csv_table['Gap_SES_Mean'] = [np.nan] + [gap_ses_mean[m] for m in target_metrics]
csv_table['Gap_SES_SE'] = [np.nan] + [gap_ses_se[m] for m in target_metrics]

csv_dir = f"{output_dir}/descriptive_statistics_summary.csv"
csv_table.to_csv(csv_dir, index=False)

print(f"LaTeX table saved to: {latex_path}")
print(f"CSV table saved to: {csv_dir}")
print("\nAnalysis complete!")

LaTeX table saved to: ../outputs/tables/descriptive_statistics_summary.tex
CSV table saved to: ../outputs/tables/descriptive_statistics_summary.csv

Analysis complete!
